# Ejercicios 09. FFT en 2D: imágenes en frecuencia

Antes de empezar:

1. Repasa el notebook de teoría `notebooks/09_fft_2d.ipynb`.
2. Ejecuta la celda de preparación, que es la primera celda de código.
3. Busca las marcas `COMPLETA` y cambia los `...` por tu código.
4. Ejecuta las celdas de verificación. Imprimen "Correcto" si tu resultado coincide y "Revisa" si no.
5. Contesta las preguntas de reflexión con tus propias palabras. Es la parte más importante.

Si una celda marca error, lee el último renglón del mensaje; casi siempre ahí está la explicación.
Por ejemplo, `NameError` quiere decir que usaste una variable que no existe o que no ejecutaste
una celda anterior.

In [ ]:
# Celda de preparación. Ejecútala antes que cualquier otra (Shift + Enter).
import sys
import os

# Si estás en Google Colab, esta parte descarga el material del curso.
# En tu computadora no hace nada.
URL_REPO = "https://github.com/TU_USUARIO/TU_REPOSITORIO.git"
if "google.colab" in sys.modules and not os.path.exists("../funciones_pds"):
    if not os.path.exists("/content/pds"):
        os.system(f"git clone -q {URL_REPO} /content/pds")
    os.chdir("/content/pds/notebooks")   # así funcionan las rutas "../datos/..."

sys.path.append("..")            # para que Python encuentre la carpeta funciones_pds

import numpy as np               # cálculos con arreglos
import matplotlib.pyplot as plt  # gráficas
import funciones_pds as pds      # la biblioteca del curso

plt.rcParams["figure.figsize"] = (9, 3)   # tamaño por defecto de las figuras
print("Bibliotecas cargadas. Ya puedes continuar.")

## Ejercicio 1. Predecir el espectro

Para una imagen $128\times128$ con **10 ciclos horizontales** y **0 verticales**:
a) Predice en qué pixeles (fila, columna) del espectro **centrado** aparecerán los dos puntos (el centro está en (64, 64)).
b) Crea la imagen (puedes copiar `imagen_coseno` del notebook de teoría), calcula el espectro y encuentra los dos máximos.

In [ ]:
def imagen_coseno(tamano, ciclos_horizontales, ciclos_verticales):
    m = np.arange(tamano)
    n = np.arange(tamano)
    imagen = np.zeros((tamano, tamano))
    for i in range(tamano):
        for j in range(tamano):
            fase = 2 * np.pi * (ciclos_verticales * m[i] + ciclos_horizontales * n[j]) / tamano
            imagen[i, j] = np.cos(fase)
    return imagen

puntos_predichos = [(..., ...), (..., ...)]       # COMPLETA: (fila, columna) de cada punto

img = imagen_coseno(128, 10, 0)
F_centrada, espectro_log = ...                    # COMPLETA
magnitud = np.abs(F_centrada)

# Encontrar los dos pixeles con mayor magnitud
puntos_encontrados = []
copia = magnitud.copy()
for repeticion in range(2):
    posicion_plana = np.argmax(copia)                          # posición en el arreglo "aplanado"
    fila, columna = np.unravel_index(posicion_plana, copia.shape)
    puntos_encontrados.append((int(fila), int(columna)))
    copia[fila, columna] = 0                                   # lo borramos para encontrar el siguiente
print("Encontrados:", sorted(puntos_encontrados))

pds.mostrar_imagen(espectro_log, titulo="espectro")
plt.show()

In [ ]:
pds.comprobar(sorted(puntos_predichos) == sorted(puntos_encontrados), "tu predicción coincide con los máximos")

## Ejercicio 2. Dirección de las franjas

Crea 3 imágenes $128\times128$: a) 6 ciclos verticales; b) 6 horizontales y 6 verticales; c) 6 horizontales y −6 verticales.
Muestra cada imagen junto a su espectro (acercamiento al centro). ¿Qué relación hay entre la dirección de las franjas y la de los puntos?

In [ ]:
casos = [(0, 6), (6, 6), (6, -6)]
figura, ejes = plt.subplots(2, 3, figsize=(13, 8))
for c in range(len(casos)):
    ch, cv = casos[c]
    img = ...                                       # COMPLETA
    _, espectro_log = ...                           # COMPLETA
    pds.mostrar_imagen(img, titulo=f"{ch} horiz., {cv} vert.", ax=ejes[0, c])
    pds.mostrar_imagen(espectro_log, titulo="espectro", ax=ejes[1, c])
    ejes[1, c].set_xlim(44, 84)
    ejes[1, c].set_ylim(84, 44)
plt.tight_layout()
plt.show()

**Reflexión:** describe la relación entre la orientación de las franjas y la posición de los puntos del espectro.

Esta es la función de filtrado pasa-bajas del notebook de teoría. **Ejecútala** (no hay que modificarla):

In [ ]:
def filtrar_imagen_pasabajas(imagen, radio):
    '''
    Filtro pasa-bajas IDEAL: conserva sólo las frecuencias dentro de un
    círculo de radio 'radio' alrededor del centro del espectro.
    Regresa la imagen filtrada y la máscara.
    '''
    filas = imagen.shape[0]
    columnas = imagen.shape[1]

    # Paso 1: FFT 2D centrada
    F_centrada, _ = pds.espectro_2d(imagen)

    # Paso 2: máscara = 1 dentro del círculo, 0 fuera
    centro_fila = filas // 2
    centro_columna = columnas // 2
    mascara = np.zeros((filas, columnas))
    for i in range(filas):
        for j in range(columnas):
            distancia = np.sqrt((i - centro_fila) ** 2 + (j - centro_columna) ** 2)
            if distancia <= radio:
                mascara[i, j] = 1.0

    # Paso 3: multiplicar espectro por máscara
    F_filtrada = F_centrada * mascara

    # Paso 4: deshacer el shift y regresar al espacio con la IFFT 2D
    F_sin_shift = np.fft.ifftshift(F_filtrada)
    imagen_filtrada = np.real(np.fft.ifft2(F_sin_shift))

    return imagen_filtrada, mascara

## Ejercicio 3. Pasa-bajas y energía

Filtra la foto con radios $r = 5, 15, 40$ y 100. Para cada uno, calcula qué **porcentaje de la energía** de la imagen
se conserva (`pds.energia` funciona con matrices si las aplanas: `imagen.flatten()`). Muestra las 4 imágenes.

In [ ]:
foto = pds.cargar_imagen("../datos/grace_hopper.jpg")
E_total = pds.energia(foto.flatten())
radios = [5, 15, 40, 100]
porcentajes = []

figura, ejes = plt.subplots(1, 4, figsize=(16, 4.5))
for i in range(len(radios)):
    filtrada, mascara = ...                        # COMPLETA
    porcentaje = ...                               # COMPLETA
    porcentajes.append(porcentaje)
    pds.mostrar_imagen(filtrada, titulo=f"r = {radios[i]}: {porcentaje:.2f}% energía", ax=ejes[i])
plt.tight_layout()
plt.show()

In [ ]:
pds.comprobar(porcentajes[0] > 70, "con r = 5 (menos del 0.03% de los coeficientes) se conserva más del 70% de la energía")
pds.comprobar(porcentajes[1] > 90, "con r = 15 se conserva más del 90% de la energía")
pds.comprobar(porcentajes == sorted(porcentajes), "el porcentaje crece con el radio")

**Reflexión:** con un círculo muy pequeño se conserva casi toda la energía, pero la imagen se ve muy mal. ¿Qué te dice esto? ¿Cómo se relaciona con la compresión JPEG?

## Ejercicio 4. Escribe un filtro pasa-altas

Escribe la función `filtrar_imagen_pasaaltas(imagen, radio)` (copia y modifica `filtrar_imagen_pasabajas` de arriba: la máscara
debe valer **0 dentro** del círculo y **1 fuera**). Aplícala a la imagen de prueba con $r = 10$. ¿Qué queda?

In [ ]:
def filtrar_imagen_pasaaltas(imagen, radio):
    '''Conserva sólo las frecuencias FUERA de un círculo de radio "radio".'''
    # COMPLETA
    ...
    return imagen_filtrada, mascara

prueba = pds.cargar_imagen("../datos/imagen_prueba.png")
altas, mascara_pa = filtrar_imagen_pasaaltas(prueba, 10)

figura, ejes = plt.subplots(1, 3, figsize=(14, 4.5))
pds.mostrar_imagen(prueba, titulo="original", ax=ejes[0])
pds.mostrar_imagen(mascara_pa, titulo="máscara pasa-altas", ax=ejes[1])
pds.mostrar_imagen(np.abs(altas), titulo="pasa-altas", ax=ejes[2])
plt.tight_layout()
plt.show()

In [ ]:
bajas, mascara_pb = filtrar_imagen_pasabajas(prueba, 10)
pds.comprobar_iguales(mascara_pa + mascara_pb, np.ones(prueba.shape), "las máscaras son complementarias")
pds.comprobar_iguales(altas + bajas, prueba, "pasa-bajas + pasa-altas = imagen original")

## Ejercicio 5. Quitar ruido periódico de una foto

Una cámara con interferencia eléctrica produce franjas en la imagen. Las vamos a quitar en el dominio de la frecuencia
con un filtro **rechaza-banda** (*notch*): poner en cero **sólo** los puntos del espectro donde está el ruido.

1. Agrega a la foto ruido periódico: $0.25\cos(2\pi \cdot 40\, n / N_{col})$ (40 ciclos horizontales).
2. Mira el espectro: ¿dónde aparecen los dos puntos del ruido?
3. Pon en cero un cuadrito de $5\times5$ alrededor de cada punto.
4. Regresa al espacio y compara con la original.

In [ ]:
foto = pds.cargar_imagen("../datos/grace_hopper.jpg")
filas, columnas = foto.shape

# Paso 1: ruido periódico (la misma fila de ruido se repite en todas las filas)
n = np.arange(columnas)
fila_ruido = 0.25 * np.cos(2 * np.pi * 40 * n / columnas)
ruido = np.zeros((filas, columnas))
for i in range(filas):
    ruido[i, :] = fila_ruido
con_ruido = foto + ruido

# Paso 2: espectro
F_centrada, espectro_log = pds.espectro_2d(con_ruido)
centro_f = filas // 2
centro_c = columnas // 2
puntos_ruido = [(centro_f, ...), (centro_f, ...)]        # COMPLETA: columnas de los dos puntos

# Paso 3: filtro notch
mascara = np.ones((filas, columnas))
for (pf, pc) in puntos_ruido:
    # COMPLETA: pon ceros en mascara[pf-2 : pf+3, pc-2 : pc+3]
    ...

# Paso 4: regresar al espacio
limpia = ...                                              # COMPLETA

_, espectro_limpio = pds.espectro_2d(limpia)
figura, ejes = plt.subplots(2, 2, figsize=(11, 10))
pds.mostrar_imagen(con_ruido, titulo="con ruido periódico", ax=ejes[0, 0])
pds.mostrar_imagen(espectro_log, titulo="espectro con ruido", ax=ejes[0, 1])
pds.mostrar_imagen(limpia, titulo="filtrada", ax=ejes[1, 0])
pds.mostrar_imagen(espectro_limpio, titulo="espectro filtrado", ax=ejes[1, 1])
plt.tight_layout()
plt.show()

In [ ]:
error_antes = np.mean(np.abs(con_ruido - foto))
error_despues = np.mean(np.abs(limpia - foto))
print(f"Error promedio antes: {error_antes:.4f}   después: {error_despues:.4f}")
pds.comprobar(error_despues < 0.1 * error_antes, "el filtro elimina más del 90% del error")

**Reflexión:** ¿por qué este ruido sería **muy difícil** de quitar con un pasa-bajas, pero es muy fácil en frecuencia con un *notch*?

## Ejercicio 6 (reto). ¿Qué es más importante: la magnitud o la fase?

Recorta de la foto un cuadrado de $256\times256$ (`foto[100:356, 100:356]`) y usa también la imagen de prueba ($256\times256$).
Crea dos imágenes "híbridas":
- **A**: magnitud de la foto + fase de la imagen de prueba.
- **B**: magnitud de la imagen de prueba + fase de la foto.

Recuerda que $X = |X|\,e^{j\angle X}$ (`np.abs` y `np.angle`). ¿A cuál se parece cada híbrida?

In [ ]:
img1 = foto[100:356, 100:356]
img2 = pds.cargar_imagen("../datos/imagen_prueba.png")

X1 = np.fft.fft2(img1)
X2 = np.fft.fft2(img2)

hibrida_A = ...         # COMPLETA: np.real(np.fft.ifft2( |X1| · e^{j·fase(X2)} ))
hibrida_B = ...         # COMPLETA

figura, ejes = plt.subplots(1, 4, figsize=(16, 4.5))
pds.mostrar_imagen(img1, titulo="foto", ax=ejes[0])
pds.mostrar_imagen(img2, titulo="prueba", ax=ejes[1])
pds.mostrar_imagen(hibrida_A, titulo="|foto| + fase(prueba)", ax=ejes[2])
pds.mostrar_imagen(hibrida_B, titulo="|prueba| + fase(foto)", ax=ejes[3])
plt.tight_layout()
plt.show()

**Reflexión:** ¿qué información guarda la fase y cuál la magnitud?